# Baseline de los mercados Financieros

En el modelado de series temporales financieras diarias (Como lo puede ser el petroleo o el tipo de cambio), el modelo base **(Baseline)** oir excelencia no es un algoritmo necesario de **Machine Learning**. Un concepto estadistico llamado **Naive Forest** (Pronostico Ingenuo) o **Random Walk**

Este modelo asume que los mercados son tan eficientes que el mejor predictor para el precio de mañana es simplemente el precio de hoy. Matematicamente es expresado de la siguiente manera: $\hat{y}_{t+1} = y_t$.

Cualquier modelo complejo como (LTSM, XGBoost, ARIMA) que se intente construir despues, debe de superar este sencillo calculo. Si un algoritmo AI no puede superar al modelo Naive, significa que no esta aprendiento nada util y solo esta agregando ruido

In [8]:
# ==========================================
# 1. IMPORT AND DATA LOADING
# ==========================================
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import mean_absolute_error, mean_squared_error

sns.set_theme(style="whitegrid")

print("Loading the energy panel data...")
df = pd.read_csv('../data/energy_panel.csv')
df['date'] = pd.to_datetime(df['date'])

Loading the energy panel data...


In [9]:
# ==========================================
# 2. PREPARATION OF THE TIME SERIES (BRENT)
# ==========================================
# Filter only the Brent series
df_brent = df[df['series_id'] == 'brent'].copy()
df_brent = df_brent.sort_values('date').set_index('date')

# Eliminate unnecessary columns to have only the value
df_brent = df_brent[['value']].rename(columns={'value': 'actual_price'})

In [10]:
# ==========================================
# 3. CREATION OF THE NAÏVE MODEL (BASELINE)
# ==========================================
# The Naïve model says: "The prediction for today is the actual price from yesterday"
print("Generating Naïve predictions (1-day shift)...")
df_brent['prediction_naive'] = df_brent['actual_price'].shift(1)

# Eliminate the first row that will have NaN due to the shift
df_brent.dropna(inplace=True)

Generating Naïve predictions (1-day shift)...


In [11]:
# ==========================================
# 4. EVALUATION OF THE DAILY BASELINE
# ==========================================
mae_naive = mean_absolute_error(df_brent['actual_price'], df_brent['prediction_naive'])
rmse_naive = np.sqrt(mean_squared_error(df_brent['actual_price'], df_brent['prediction_naive']))

print(f"\n--- PERFORMANCE OF THE NAÏVE MODEL (DAILY BRENT) ---")
print(f"Total of days evaluated: {len(df_brent)}")
print(f"MAE  (Error Absoluto Medio): {mae_naive:.4f} USD per barrel")
print(f"RMSE (Error Cuadrático Medio): {rmse_naive:.4f} USD per barrel")


--- PERFORMANCE OF THE NAÏVE MODEL (DAILY BRENT) ---
Total of days evaluated: 9957
MAE  (Error Absoluto Medio): 0.8188 USD per barrel
RMSE (Error Cuadrático Medio): 1.3541 USD per barrel


## Reto de Random Walk

Equivocarse en promedio de 81 centavos == 0.8188 USD al predecir el precio del barril al dia siguiente usando la regla de "el precio de mñana seguira siendo al de hoy" demuestra la **teoria del Random Walk**. Los mercados financieros son tan eficientes absorbiendo informacion dia a dia, que el precio actual ya contiene casi todo lo que se sabe sobre el activo.

El gran reto es ahora intentar superar esos $0.81 centavos construyendo un modelo avanzado de machine learning

## Feature Engineering and LightGBM Daily

Nuevamente se crearan rezagos diarios (lags) y medias moviles (rolling means) para que LightGBM detecte tendencias a corto plazo

In [12]:
# ==========================================
# 5. FEATURE ENGINEERING (DAILY)
# ==========================================
print("Creating temporal variables for Brent crude...")
df_features = df_brent.copy()

# 1. Lags of 1 to 5 days (Entire previous week)
for i in range(1, 6):
    df_features[f'lag_{i}'] = df_features['actual_price'].shift(i)

# 2. Rolling means for trend detection
# We use .shift(1) to avoid data leakage and include today's price in the rolling mean
df_features['rolling_mean_5'] = df_features['actual_price'].shift(1).rolling(window=5).mean()
df_features['rolling_mean_21'] = df_features['actual_price'].shift(1).rolling(window=21).mean()

# Drop the first 21 rows that have NaN values due to the rolling calculations
df_features.dropna(inplace=True)

Creating temporal variables for Brent crude...


In [13]:
# ==========================================
# 6. SEPARATION AND TRAINING (LightGBM)
# ==========================================
from lightgbm import LGBMRegressor

y = df_features['actual_price']
X = df_features.drop(columns=['actual_price', 'prediction_naive'])

# Cronological Division (80% Train, 20% Test)
train_size = int(len(df_features) * 0.8)
X_train, X_test = X.iloc[:train_size], X.iloc[train_size:]
y_train, y_test = y.iloc[:train_size], y.iloc[train_size:]

print("Training LightGBM model for daily prediction...")
model_lgb_diario = LGBMRegressor(
    n_estimators=100,
    learning_rate=0.05,
    max_depth=4,
    random_state=42,
    extra_trees=True
)
model_lgb_diario.fit(X_train, y_train)

Training LightGBM model for daily prediction...
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000414 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 1785
[LightGBM] [Info] Number of data points in the train set: 7948, number of used features: 7
[LightGBM] [Info] Start training from score 45.830515
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] 

,max_depth,4
,learning_rate,0.05
,random_state,42
,extra_trees,True
,boosting_type,'gbdt'
,num_leaves,31
,n_estimators,100
,subsample_for_bin,200000
,objective,None
,class_weight,None
,min_split_gain,0.0


In [14]:
# ==========================================
# 7. EVALUATION AND COMPARISON
# ==========================================
pred_lgb = model_lgb_diario.predict(X_test)
mae_lgb = mean_absolute_error(y_test, pred_lgb)

# Re-compute MAE for the naive model strictly on the test set
mae_naive_test = mean_absolute_error(y_test, df_features['prediction_naive'].iloc[train_size:])

print(f"\n--- COMPARATIVE in the TEST SET (FUTURE) ---")
print(f"MAE Naïve (Baseline): {mae_naive_test:.4f} USD")
print(f"MAE LightGBM:         {mae_lgb:.4f} USD")


--- COMPARATIVE in the TEST SET (FUTURE) ---
MAE Naïve (Baseline): 1.4141 USD
MAE LightGBM:         2.4196 USD


# Una vez mas el modelo de machine learning no logro superar sus cometidos en este caso el modelo Naive

¿Que tan predecibles son las series de mercado? la respuesta tecnica es que **son casi imposibles de predecir a corto plazo con modelos no lineales**


Esto se explica mediante la Teoría del Paseo Aleatorio (Random Walk) y la Hipótesis de Mercados Eficientes:

- Los mercados financieros diarios absorben noticias e información al instante.

- El precio de "hoy" ya refleja todas las expectativas públicas sobre el activo.

- Al intentar forzar a LightGBM a predecir el mañana, el algoritmo termina sobreajustándose al "ruido" estadístico en lugar de encontrar una señal predictiva real, duplicando el error.

**Note:** Comprobé empíricamente que intentar predecir el mercado diario con ML complejo añade más error que valor, validando la teoría del paseo aleatorio. Por lo tanto, para hacer un 'nowcast' el día 15 del mes, la estrategia más robusta es proyectar los 15 días faltantes utilizando el último precio observado (Naïve) o una media móvil simple. Luego, agrupo ese mes completo y alimento mi Regresión Ridge (de la Parte 2), la cual demostró ser superior para mapear esa energía hacia la inflación al consumidor porque extrapola mejor en escenarios de alta inflación.